Retail Streaming Analytics

Notebook: 02_Bronze_To_Silver

Purpose:
- Read Bronze transactions
- Apply quality checks
- Remove duplicates
- Create Silver transactions table

Transformations:
- Remove null business keys
- Filter invalid quantity
- Filter invalid unit price
- Filter invalid gross amount
- Remove Kafka metadata columns
- Add silver_load_ts
Source:
retailanalytics.bronze.transaction_raw

Target:
retailanalytics.silver.silver_transaction

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp
)

In [0]:
# loading bronze table
bronze_stream = (
    spark.readStream
    .table(
        "retailanalytics.bronze.transaction_raw"
    )
)

In [0]:
# Silver Transformation
silver_stream = (
    bronze_stream
    .filter(col("transaction_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("store_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("unit_price") > 0)
    .filter(col("gross_amount") > 0)
    .drop("topic", "partition", "offset")
    .withColumn(
        "silver_load_ts",
        current_timestamp()
    )
)

In [0]:
silver_checkpoint = (
    "/Volumes/retailanalytics/secrets/"
    "kafkacerts/checkpoints/"
    "silver_transaction"
)

In [0]:
(
    silver_stream.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        silver_checkpoint
    )
    .trigger(availableNow=True)
    .toTable(
        "retailanalytics.silver.silver_transaction"
    )
)

In [0]:
silver_df = spark.table(
    "retailanalytics.silver.silver_transaction"
)

print(
    f"Silver Count: {silver_df.count()}"
)

display(
    silver_df.limit(10)
)